# 01 — Data Collection

Downloads daily OHLCV data for 9 stocks from 2021 to 2026 using yfinance. Saves one CSV per stock to `data/raw/`.

In [1]:
!pip install yfinance pandas --quiet

In [2]:
import os
import yfinance as yf
import pandas as pd

BASE_DIR   = "."
OUTPUT_DIR = os.path.join(BASE_DIR, "data/raw")

TICKERS    = ["AAPL", "MSFT", "JPM", "GS", "JNJ", "PFE", "SPOT", "AMZN", "TSLA"]
START_DATE = "2021-01-01"
END_DATE   = "2026-04-24"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output folder:", os.path.abspath(OUTPUT_DIR))

Output folder: /Users/rutuja/Documents/CS6140 Machine Learning/ml project/Stock-Price-Prediction/data/raw


## 1. Download OHLCV data

`auto_adjust=True` adjusts prices for stock splits and dividends. Without this, a stock split looks like an overnight price crash and flips the classification label.

In [3]:
failed = []

for ticker in TICKERS:
    try:
        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            actions=False,
            progress=False,
        )

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
        df.index = pd.to_datetime(df.index)
        df.index.name = "Date"

        path = os.path.join(OUTPUT_DIR, f"{ticker}.csv")
        df.to_csv(path)

        print(f"{ticker:<5}  {len(df)} rows  "
              f"({df.index.min().date()} to {df.index.max().date()})  "
              f"nulls={df.isnull().sum().sum()}")

    except Exception as e:
        print(f"{ticker:<5}  FAILED: {e}")
        failed.append(ticker)

if failed:
    print(f"Failed tickers: {failed}")
else:
    print("All 9 tickers downloaded successfully.")

AAPL   1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
MSFT   1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
JPM    1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
GS     1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
JNJ    1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
PFE    1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
SPOT   1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
AMZN   1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
TSLA   1332 rows  (2021-01-04 to 2026-04-23)  nulls=0
All 9 tickers downloaded successfully.


## 2. Verify date alignment

All 9 stocks must share the same trading dates. Misaligned dates cause NaN rows when features are merged later.

In [4]:
date_sets = {}
for ticker in TICKERS:
    path = os.path.join(OUTPUT_DIR, f"{ticker}.csv")
    idx  = pd.read_csv(path, index_col="Date", parse_dates=True).index
    date_sets[ticker] = set(idx)

reference       = TICKERS[0]
reference_dates = date_sets[reference]
all_aligned     = True

for ticker, dates in date_sets.items():
    diff = reference_dates.symmetric_difference(dates)
    if diff:
        print(f"{ticker} has {len(diff)} misaligned date(s): {sorted(diff)}")
        all_aligned = False

if all_aligned:
    print(f"All 9 tickers share the same {len(reference_dates)} trading dates.")

All 9 tickers share the same 1332 trading dates.


## 3. Sanity check

In [5]:
sample = pd.read_csv(os.path.join(OUTPUT_DIR, "AAPL.csv"), index_col="Date", parse_dates=True)
print("Shape:", sample.shape)
display(sample.head(3))
display(sample.tail(3))

Shape: (1332, 5)


,Open,High,Low,Close,Volume
Date,,,,,
2021-01-04,129.853870,129.941395,123.279481,125.856720,143301900
2021-01-05,125.350965,128.122717,124.903590,127.412750,97664900
2021-01-06,124.213113,127.451681,122.909903,123.123863,155088000


,Open,High,Low,Close,Volume
Date,,,,,
2026-04-21,271.500000,272.799988,265.399994,266.170013,50209800
2026-04-22,267.820007,273.739990,266.869995,273.170013,43249200
2026-04-23,275.045013,275.769989,271.649994,273.829010,22590029


## 4. Save metadata

In [6]:
rows = []
for ticker in TICKERS:
    path = os.path.join(OUTPUT_DIR, f"{ticker}.csv")
    df   = pd.read_csv(path, index_col="Date", parse_dates=True)
    rows.append({
        "Ticker": ticker,
        "Rows"  : len(df),
        "Start" : df.index.min().date(),
        "End"   : df.index.max().date(),
        "Nulls" : df.isnull().sum().sum(),
    })

metadata = pd.DataFrame(rows)
metadata.to_csv(os.path.join(OUTPUT_DIR, "_metadata.csv"), index=False)
display(metadata)
print("Notebook 01 complete — raw data saved to data/raw/")

,Ticker,Rows,Start,End,Nulls
0,AAPL,1332,2021-01-04,2026-04-23,0
1,MSFT,1332,2021-01-04,2026-04-23,0
2,JPM,1332,2021-01-04,2026-04-23,0
3,GS,1332,2021-01-04,2026-04-23,0
4,JNJ,1332,2021-01-04,2026-04-23,0
5,PFE,1332,2021-01-04,2026-04-23,0
6,SPOT,1332,2021-01-04,2026-04-23,0
7,AMZN,1332,2021-01-04,2026-04-23,0
8,TSLA,1332,2021-01-04,2026-04-23,0


Notebook 01 complete — raw data saved to data/raw/
